# PFX Extractor Test V3 — Bandit v2 / DnR v3, traitement dans Colab

**Révision UMID du 6 octobre 2026 — identifiants neufs et uniques vérifiés, protocole de livraison 2.** Recharger ce fichier dans Colab et redémarrer le frontend. Aucun changement de traitement sonore. Valider le nouvel import dans une session Pro Tools neuve pour éviter les anciennes associations de médias.

**Variante du 5 octobre 2026 : scènes indépendantes, aucun alignement BOOM/LAV, pré-nettoyage adaptatif 40 %, finition sur la sortie Bandit, retrait humain non cumulatif et bodytalk maximal +4 dB avec marge locale.**

Chaque WAV reste une scène indépendante. BOOM et LAV ne sont jamais mélangés, déplacés ou regroupés. Les WAV doivent être mono 48 kHz avec timecode BWF, entre 1024 et 16777215 échantillons (limite actuelle du constructeur PTX). Les noms doivent contenir BOOM ou LAV, pas les deux; nombres et préfixes libres. Si les deux micros sont présents, les comptes doivent être égaux. Validation avant calculs.

Le gain bodytalk ne produit pas une piste de vêtements isolée : les pas/impacts détectés l'inhibent, mais des sons simultanés non détectés peuvent encore monter. Le retrait humain peut encore atténuer du Foley simultané. Les seuils restent heuristiques.

**Workflow :** ouvrir ce notebook dans une nouvelle VM Colab T4, exécuter 1 à 8 et utiliser le bouton ZIP/PTX habituel dans le **frontend local mis à jour et redémarré**. Ne pas lancer deux notebooks en parallèle. Sorties par génération dans `2_Environnements_IA/run_<identifiant>`; manifeste `PFX_delivery.json` publié complet uniquement après vérification. Le frontend refuse les lots incomplets ou les WAV dont l'empreinte/durée/timecode ne correspondent pas. Il ignore les anciens WAV restés à la racine. Les anciens rendus ne sont pas écrasés. Ne pas utiliser l'ancien frontend pour ce notebook.

Si un run a été brutalement interrompu, vérifier que sa VM ne travaille plus avant d'activer `REMPLACER_RUN_INTERROMPU`. Les anciens notebooks restent inchangés; pour revenir à leur workflow après V3, archiver/retirer explicitement le manifeste V3 de la sortie Drive (sinon il reste prioritaire).

**Confidentialité :** traitement dans Colab et Drive uniquement, sans upload audio vers une API externe. Les poids/sources publics de Bandit sont téléchargés et vérifiés. Modèle officiel multilingue Bandit v2 / DnR v3, sortie SFX; ce n'est ni DnR-nonverbal ni MVSEP.

Les tests locaux vérifient la géométrie et les contrats, pas la qualité auditive. Le défaut de collision UMID a été identifié sur le lot réel. Cette révision le corrige à l’export; une nouvelle validation manuelle dans Pro Tools reste requise.


In [ ]:
# 1. 🎛️ PANNEAU DE CONTRÔLE UTILISATEUR
#@title Ajustement de la Réduction de Bruit (Pre-Denoise)
#@markdown Glissez le curseur pour définir la force du nettoyage avant le passage de l'IA.
#@markdown Défaut Test V3 : 40 %, retenu après écoute utilisateur.
#@markdown Un réglage plus fort peut atténuer les mouvements discrets.

NIVEAU_DENOISE_POURCENTAGE = 40 #@param {type:"slider", min:0, max:100, step:1}

#@markdown ---
#@markdown **Pré-denoise adaptatif (anti-pumping)**
#@markdown Un profil de bruit stationnaire matche mal un bruit extérieur qui varie (vent, trafic) —
#@markdown ce mismatch peut faire "pomper" le fond sonore aux endroits où il y avait du dialogue.
#@markdown * Activé (recommandé) = profil de bruit adaptatif (`stationary=False`), plus robuste sur l'extérieur
#@markdown * Désactivé = comportement V1.x (`stationary=True`), peut mieux convenir à un bruit de fond très stable
DENOISE_ADAPTATIF = True #@param {type:"boolean"}

#@markdown ---
#@markdown **Séparateur Test V3 : Bandit v2 multilingue / sortie SFX**
#@markdown Un modèle cinéma, sans mélange MDX/RoFormer.
#@markdown Fenêtres 8 secondes, pas 1 seconde, batch 1; calcul float32.
#@markdown ---

#@markdown **Retrait des sons humains (YAMNet)**
#@markdown Atténue voix, vocalisations, respirations, sons physiologiques et applaudissements.
#@markdown Le masque utilise des indices YAMNet exacts, un seuil calibré et une enveloppe temporelle.
#@markdown * 75% = défaut V3.0 équilibré (plage conseillée : 70-80%)
DUCK_DEPTH_HUMAIN = 75 #@param {type:"slider", min:0, max:100, step:5}

#@markdown **Retrait prioritaire soupirs / chuchotements / respirations**
#@markdown 95% = jusqu’à environ -26 dB lorsque la détection est maximale, même avec protection PFX.
#@markdown 0% désactive uniquement le renforcement des souffles.
#@markdown Peut atténuer les vêtements et manipulations simultanés.
DUCK_DEPTH_SOUFFLES = 95 #@param {type:"slider", min:0, max:100, step:5}

#@markdown **Retrait prioritaire des pleurs / sanglots / gémissements**
#@markdown 95% = jusqu'à -26 dB; combinaison par maximum avec les résidus, jamais multiplication des retraits.
#@markdown Peut atténuer le foley simultané. 0% désactive les deux renforts pleurs, pas le retrait humain général.
DUCK_DEPTH_PLEURS = 95 #@param {type:"slider", min:0, max:100, step:5}

#@markdown **V3.4 : profil respiratoire renforcé**
#@markdown Ajoute Sniff, Snort, Gasp et Pant; sensibilité et marges accrues.
#@markdown Désactiver ce profil ET mettre les résidus à 0% retrouve V3.3.0.
PROFIL_RESPIRATOIRE_RENFORCE = True #@param {type:"boolean"}

#@markdown **V3.4 : retrait supplémentaire des résidus humains**
#@markdown Analyse YAMNet complémentaire du SFX non atténué, avant un retrait unique.
#@markdown 90% = jusqu’à -20 dB supplémentaires localement; 0% désactive cette analyse.
#@markdown Peut aussi réduire le foley simultané. Aucun nouveau passage RoFormer/MDX.
DUCK_DEPTH_RESIDUS = 90 #@param {type:"slider", min:0, max:95, step:5}

#@markdown **V3.5 : gain d’écoute du détecteur de résidus (dB)**
#@markdown Jusqu’à +18 dB sur une copie de chaque fenêtre, limitée en crête.
#@markdown Le volume du rendu ne monte pas. Peut déclencher davantage de faux positifs.
#@markdown 0 dB retrouve exactement l’analyse V3.4; résidus à 0% saute les deux analyses.
GAIN_ANALYSE_RESIDUS_DB = 18 #@param {type:"slider", min:0, max:24, step:3}

#@markdown ---
#@markdown **Retrait prioritaire des bruits de bouche**
#@markdown Mastication/morsure/gargarisme + smacks/clics/pops accompagnés d’indices humains proches.
#@markdown 95% = jusqu’à -26 dB; peut retirer du foley simultané. 0% désactive cet ajout.
DUCK_DEPTH_BOUCHE = 95 #@param {type:"slider", min:0, max:100, step:5}

#@markdown **Test V3 : gain temporel des mouvements de vêtements / bodytalk (dB)**
#@markdown Défaut : jusqu’à +4 dB sur froissements/frottements/zipper (+0,5 dB depuis V3.5.1).
#@markdown Les classes footsteps, tap et slap ne déclenchent pas ce gain.
#@markdown Freiné par les indices humains/ambiants et la marge disponible avant saturation.
#@markdown Bouche à 0% et bodytalk à 0 dB désactivent les ajouts V3.3, pas ceux de V3.4.
GAIN_BODYTALK_DB = 4.0 #@param {type:"slider", min:0, max:4, step:0.5}

#@markdown ---
#@markdown **Retrait des ambiances**
#@markdown Réduit nature/météo, trafic, roomtone, bruit continu et bourdonnements.
#@markdown * 70% = défaut V3.0; le pré-denoise agit déjà en amont (plage conseillée : 60-75%)
DUCK_DEPTH_AMBIANCE = 70 #@param {type:"slider", min:0, max:100, step:5}

#@markdown ---
#@markdown **Retrait du contenu hors PFX**
#@markdown Réduit musique, TV/radio, véhicules, sirènes et graves/vibrations.
#@markdown * 85% = retrait fort sans hard gate (plage conseillée : 80-90%)
DUCK_DEPTH_HORS_PFX = 85 #@param {type:"slider", min:0, max:100, step:5}

#@markdown ---
#@markdown **Protection Production FX**
#@markdown Protège pas, tissu, manipulations, props, impacts et outils contre le pré-denoise et les
#@markdown faux positifs YAMNet. La restauration brute est plafonnée pour ne jamais annuler le nettoyage.
#@markdown * 0% = Aucune protection additionnelle
#@markdown * 70% = défaut V3.0 équilibré (plage conseillée : 60-75%)
PROTECTION_PFX = 70 #@param {type:"slider", min:0, max:90, step:5}

print(f"✅ Niveau de réduction de bruit réglé à : {NIVEAU_DENOISE_POURCENTAGE}% ({'adaptatif' if DENOISE_ADAPTATIF else 'stationnaire'})")
print(f"✅ Retrait YAMNet : humains {DUCK_DEPTH_HUMAIN}% | ambiance {DUCK_DEPTH_AMBIANCE}% | hors-PFX {DUCK_DEPTH_HORS_PFX}%")
print(f"✅ Protection PFX : {'Désactivée' if PROTECTION_PFX == 0 else f'{PROTECTION_PFX}%'}")

print("✅ Test V3 : Bandit v2 multilingue — SFX — 48 kHz mono — calcul dans Colab")

#@markdown Activer seulement après avoir confirmé l’arrêt d’un ancien run interrompu.
REMPLACER_RUN_INTERROMPU = False #@param {type:"boolean"}


In [ ]:
# 2. DÉPENDANCES — pas de client MVSEP, pas de service de traitement externe
import os
import sys
import subprocess
from importlib.metadata import version as package_version
os.environ['WANDB_DISABLED'] = 'true'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['DO_NOT_TRACK'] = '1'

# Conserver la paire torch/torchaudio du runtime Colab; ne pas mélanger CPU/CUDA.
import torch
import torchaudio
if torch.__version__.split('+')[0] != torchaudio.__version__.split('+')[0]:
    raise RuntimeError("Versions torch/torchaudio incompatibles : redémarrer une session Colab propre.")
if not torch.cuda.is_available():
    raise RuntimeError("Activez un GPU dans Exécution > Modifier le type d'exécution avant de continuer.")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pytorch-lightning==2.5.1', 'librosa==0.11.0', 'soundfile==0.13.1',
    'noisereduce==3.0.3', 'scipy', 'PyYAML==6.0.2', 'requests',
    'tensorflow-hub==0.16.1'])
print(f"✅ GPU : {torch.cuda.get_device_name(0)} | torch {torch.__version__}")
print(f"✅ Lightning {package_version('pytorch-lightning')} | noisereduce {package_version('noisereduce')}")


In [ ]:
# 3. DRIVE — mêmes entrées/sorties que V3.5.1, temporaires isolés
import os
from datetime import datetime, timezone
from uuid import uuid4
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = "/content/drive/MyDrive/PFX_Extractor"
FOLDER_IN = os.path.join(DRIVE_BASE, "1_Bruts_vers_Colab")
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
FOLDER_OUT = os.path.join(DRIVE_BASE, "2_Environnements_IA")
FOLDER_MODEL_CACHE = os.path.join(DRIVE_BASE, "0_Model_Cache", "Bandit_v2")
TEMP_ROOT = os.path.join("/content/pfx_test_v3", RUN_ID)
FOLDER_ALIGNED = os.path.join(TEMP_ROOT, "aligned")
FOLDER_MERGED = os.path.join(TEMP_ROOT, "merged")
FOLDER_DENOISED = os.path.join(TEMP_ROOT, "denoised")
FOLDER_AI_STEMS = os.path.join(TEMP_ROOT, "stems")
FOLDER_TC_READY = os.path.join(TEMP_ROOT, "tc_ready")
WORKFLOW_COMPLETE = False
for folder in [FOLDER_IN, FOLDER_OUT, FOLDER_MODEL_CACHE, FOLDER_ALIGNED,
               FOLDER_MERGED, FOLDER_DENOISED, FOLDER_AI_STEMS, FOLDER_TC_READY]:
    os.makedirs(folder, exist_ok=True)
print(f"📁 Sortie habituelle du frontend : {FOLDER_OUT}")
print("Test V3 : livraison vérifiée dans un dossier propre à chaque run; frontend mis à jour requis.")


In [ ]:
# 4. BANDIT V2 — sources officielles épinglées, poids vérifiés
import gc
import shutil
import re
import subprocess
import soundfile as sf
import noisereduce as nr
from scipy import signal

MIN_SAFE_DURATION = 5.0
MIN_EXPORT_SAMPLES = 1024
"""Runtime embedded verbatim in Vtest: official Bandit v2, local inference only.

No audio is passed to a network operation. Upstream sources and model weights
are downloaded once, verified, and cached. This is NOT DnR-nonverbal or MVSEP.
"""
from pathlib import Path
import hashlib
import importlib
import sys
import time

import numpy as np

BANDIT_COMMIT = 'd5563d9031e95fdaa3e5a73d5020b9a0df61adb6'
BANDIT_CHECKPOINT_URL = 'https://zenodo.org/api/records/12701995/files/checkpoint-multi.ckpt/content'
BANDIT_CHECKPOINT_MD5 = 'fea2868787551b0cff36cfcf7c3622a3'
BANDIT_CHECKPOINT_SIZE = 446680129
BANDIT_RATE = 48000
BANDIT_STEMS = ['speech', 'music', 'sfx']
BANDIT_SOURCES = {
    'LICENSE': '261eeb9e9f8b2b4b0d119366dda99c6fd7d35c64',
    'configs/models/bandit-mus64.yaml': 'c68a3ad1756908bc23726c46e2eb7206c4ee2572',
    'src/models/base.py': '49fd4191102d9e4880d50e6f42dc1030c32086e5',
    'src/models/bandit/bandit.py': '02e0768766f979ffdd79bc280a757d9716abd22f',
    'src/models/bandit/bandsplit.py': '6bf8d4437248391caf9f741abc59fbdd1ebcfda3',
    'src/models/bandit/film.py': 'e30795332ea0e06865ea3d883767db17bb02353c',
    'src/models/bandit/maskestim.py': '8677ea6130407f54dde0265b21227c7f03bfb320',
    'src/models/bandit/tfmodel.py': '21aef03d1f0e814c20db05fe7d14f8019f07713b',
    'src/models/bandit/utils.py': 'ad4eab5d8c5b5396ed717f5b9c365a6900eddd2f',
}


def _file_digest(path, algorithm, git_blob=False):
    digest = hashlib.new(algorithm)
    if git_blob:
        digest.update(f'blob {path.stat().st_size}\0'.encode('ascii'))
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def _verified_download(url, path, expected, algorithm, size=None, git_blob=False):
    """GET only; no audio path, body, API key or user information is sent."""
    import requests
    path = Path(path)
    def valid(candidate):
        return (candidate.is_file()
                and (size is None or candidate.stat().st_size == size)
                and _file_digest(candidate, algorithm, git_blob) == expected)
    if valid(path):
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    part = path.with_name(path.name + '.part')
    for attempt in range(1, 4):
        try:
            print(f'Téléchargement public : {path.name} ({attempt}/3)')
            with requests.get(url, stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                with part.open('wb') as handle:
                    for block in response.iter_content(1024 * 1024):
                        handle.write(block)
            if not valid(part):
                raise RuntimeError(f'Intégrité incorrecte : {path.name}')
            part.replace(path)
            return path
        except Exception:
            part.unlink(missing_ok=True)
            if attempt == 3:
                raise
            time.sleep(3 * attempt)


def prepare_bandit_assets(cache_dir):
    cache = Path(cache_dir)
    package_name = 'pfx_bandit_' + BANDIT_COMMIT[:12]
    source_root = cache / package_name
    for relative, blob in BANDIT_SOURCES.items():
        _verified_download(
            f'https://raw.githubusercontent.com/kwatcharasupat/bandit-v2/{BANDIT_COMMIT}/{relative}',
            source_root / relative, blob, 'sha1', git_blob=True,
        )
    checkpoint = _verified_download(
        BANDIT_CHECKPOINT_URL, cache / 'checkpoint-multi.ckpt',
        BANDIT_CHECKPOINT_MD5, 'md5', BANDIT_CHECKPOINT_SIZE,
    )
    return source_root, checkpoint


def bandit_model_state(checkpoint):
    """Extract only the model from the original Lightning training checkpoint."""
    state = checkpoint.get('state_dict')
    if not isinstance(state, dict):
        raise RuntimeError('Checkpoint Bandit sans state_dict attendu.')
    result = {key[len('model.'):]: value for key, value in state.items()
              if key.startswith('model.')}
    if not result:
        raise RuntimeError('Checkpoint Bandit sans paramètres model.*.')
    return result


def load_bandit(cache_dir, device='cuda'):
    import torch
    import yaml
    source_root, checkpoint_path = prepare_bandit_assets(cache_dir)
    cache_parent = str(source_root.parent.resolve())
    if cache_parent not in sys.path:
        sys.path.insert(0, cache_parent)
    module = importlib.import_module(source_root.name + '.src.models.bandit.bandit')
    config = yaml.safe_load((source_root / 'configs/models/bandit-mus64.yaml').read_text())
    model = module.Bandit(fs=BANDIT_RATE, stems=list(BANDIT_STEMS), **config['kwargs'])
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    # No strict=False: an incompatible checkpoint must stop before any real batch.
    model.load_state_dict(bandit_model_state(checkpoint), strict=True)
    del checkpoint
    return model.eval().to(device)


def validate_bandit_audio(audio, sample_rate):
    audio = np.asarray(audio, dtype=np.float32)
    if sample_rate != BANDIT_RATE:
        raise ValueError(f'Vtest attend du 48000 Hz, reçu {sample_rate}.')
    if audio.ndim != 2 or audio.shape[1] != 1 or not len(audio):
        raise ValueError('Vtest attend un WAV mono non vide (N, 1).')
    if not np.isfinite(audio).all():
        raise ValueError('Audio contenant NaN ou Inf.')
    return audio


def bandit_overlap_add(audio, predict, chunk_samples=384000, hop_samples=48000):
    """Bounded GPU input, CPU overlap-add; preserve sample count and gain.

    Eight-second windows / one-second hop, as in the upstream inference config.
    Symmetric reflected context and actual window-sum normalization prevent edge
    fades, including for very short clips. Only the SFX stem is retained.
    """
    audio = validate_bandit_audio(audio, BANDIT_RATE)
    if not 0 < hop_samples <= chunk_samples or chunk_samples < 2:
        raise ValueError('Fenêtre/pas Bandit invalides.')
    n = len(audio)
    pad = chunk_samples
    padded = np.pad(audio[:, 0], (pad, pad), mode='reflect' if n > 1 else 'edge')
    window = np.hanning(chunk_samples + 1)[:-1].astype(np.float32)
    result = np.zeros(len(padded), dtype=np.float32)
    weight = np.zeros(len(padded), dtype=np.float32)
    starts = range(0, len(padded) - chunk_samples + 1, hop_samples)
    total = len(starts)
    for index, start in enumerate(starts):
        # Windows wholly outside the retained region cannot contribute.
        end = start + chunk_samples
        if end <= pad or start >= pad + n:
            continue
        chunk = np.ascontiguousarray(padded[start:end])
        estimate = np.asarray(predict(chunk), dtype=np.float32)
        if estimate.shape != (chunk_samples,) or not np.isfinite(estimate).all():
            raise RuntimeError('Sortie SFX Bandit invalide (forme/NaN/Inf). Aucun bypass.')
        result[start:end] += estimate * window
        weight[start:end] += window
        if index % 30 == 0:
            print(f'   Bandit : fenêtre {index + 1}/{total}')
    result, weight = result[pad:pad + n], weight[pad:pad + n]
    if np.any(weight <= 0):
        raise RuntimeError('Trou dans la reconstruction Bandit.')
    return (result / weight).reshape(-1, 1)


def predict_bandit_sfx(model, chunk):
    import torch
    device = next(model.parameters()).device
    tensor = torch.from_numpy(np.ascontiguousarray(chunk)).reshape(1, 1, -1).to(device)
    with torch.inference_mode():
        output = model({'mixture': {'audio': tensor}})
        estimate = output['estimates']['sfx']['audio']
        if tuple(estimate.shape) != (1, 1, len(chunk)):
            raise RuntimeError(f'Forme SFX inattendue : {tuple(estimate.shape)}')
        return estimate[0, 0].detach().float().cpu().numpy().copy()


def bandit_separate_audio(model, audio, sample_rate):
    audio = validate_bandit_audio(audio, sample_rate)
    return bandit_overlap_add(audio, lambda chunk: predict_bandit_sfx(model, chunk))



_bandit_model = None

def preflight_models():
    global _bandit_model
    if not torch.cuda.is_available():
        raise RuntimeError("Vtest nécessite un GPU Colab.")
    print("PREFLIGHT Bandit : vérification du modèle puis inférence synthétique 8 s...")
    _bandit_model = load_bandit(FOLDER_MODEL_CACHE, device='cuda')
    probe = (0.01 * np.sin(2 * np.pi * 440 * np.arange(384000) / BANDIT_RATE)).astype(np.float32)
    result = predict_bandit_sfx(_bandit_model, probe)
    if result.shape != probe.shape or not np.isfinite(result).all():
        raise RuntimeError("Échec du test d'inférence Bandit.")
    print("✅ PREFLIGHT RÉUSSI — poids stricts, sortie SFX et GPU vérifiés.")

def process_bandit_batch(files_list):
    global _bandit_model
    if _bandit_model is None:
        raise RuntimeError("Exécutez le preflight Bandit.")
    try:
        for filename in files_list:
            print(f"🎛️ Bandit SFX : {filename}")
            audio, sr = sf.read(os.path.join(FOLDER_DENOISED, filename),
                                dtype='float32', always_2d=True)
            result = bandit_separate_audio(_bandit_model, audio, sr)
            destination = os.path.join(FOLDER_AI_STEMS, os.path.splitext(filename)[0] + "_Bandit.wav")
            sf.write(destination, result, sr, subtype='FLOAT')
    finally:
        _bandit_model = None
        gc.collect()
        torch.cuda.empty_cache()

print("✅ Configuration Bandit prête. Aucun appel réseau ne reçoit d'audio.")


In [ ]:
# 5. DÉTECTEUR YAMNET — TAXONOMIE PFX EXACTE ET MASQUES SÉPARÉS
"""
Les 521 classes AudioSet sont partitionnées par indices exacts. Aucune recherche par
sous-chaîne n'est autorisée : elle confondait notamment Hum/Humming et Run avec des
classes sans rapport. Les masques humain, ambiance, hors-PFX et protection PFX ont
des seuils et des enveloppes temporelles distincts.
"""
import tensorflow as tf
# Ne pas réserver toute la VRAM avant le chargement Bandit.
for _gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_gpu, True)
import tensorflow_hub as hub
import csv

print("📥 Chargement du modèle YAMNet (détecteur d'événements sonores)...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

class_map_path = yamnet_model.class_map_path().numpy().decode('utf-8')
class_names = []
with tf.io.gfile.GFile(class_map_path) as f:
    reader = csv.DictReader(f)
    for row in reader:
        class_names.append(row['display_name'])

if len(class_names) != 521:
    raise RuntimeError(f"Taxonomie YAMNet inattendue : {len(class_names)} classes au lieu de 521.")
YAMNET_SENTINEL_NAMES = {
    353: 'Knock', 354: 'Tap', 413: 'Hammer', 454: 'Thump, thud', 460: 'Bang',
    19: 'Crying, sobbing', 20: 'Baby cry, infant cry', 21: 'Whimper', 22: 'Wail, moan',
    12: 'Whispering', 23: 'Sigh', 36: 'Breathing',
    39: 'Gasp', 40: 'Pant', 41: 'Snort', 45: 'Sniff',
    49: 'Chewing, mastication', 50: 'Biting', 51: 'Gargling',
    372: 'Zipper (clothing)', 428: 'Burst, pop', 461: 'Slap, smack',
    470: 'Rub', 473: 'Crumpling, crinkling', 481: 'Rustle', 485: 'Clicking',
    0: 'Speech', 58: 'Clapping', 62: 'Applause', 132: 'Music', 277: 'Wind',
    294: 'Vehicle', 321: 'Traffic noise, roadway noise', 390: 'Siren',
    458: 'Chorus effect', 487: 'Rumble', 490: 'Hum', 494: 'Silence',
    495: 'Sine wave', 500: 'Inside, small room', 518: 'Television',
    520: 'Field recording',
}
sentinel_mismatches = {
    index: (expected, class_names[index])
    for index, expected in YAMNET_SENTINEL_NAMES.items()
    if class_names[index] != expected
}
if sentinel_mismatches:
    raise RuntimeError(f"Ordre de taxonomie YAMNet incompatible : {sentinel_mismatches}")

def _inclusive(start, end):
    return set(range(start, end + 1))

HUMAN_CLASS_SET = (
    _inclusive(0, 45)
    | _inclusive(49, 55)
    | {59, 60, 61, 62, 63, 64, 65, 66}
)
PFX_CLASS_SET = (
    {46, 47, 48, 56, 57, 58}
    | _inclusive(348, 389)
    | {392, 393, 394}
    | _inclusive(398, 457)
    | _inclusive(459, 486)
    | {488, 489, 491, 492, 493, 498, 499}
)
AMBIENCE_CLASS_SET = (
    _inclusive(277, 293)
    | {321, 490, 507, 508, 509, 510, 513, 514, 515}
)
OUTSIDE_PFX_CLASS_SET = (
    _inclusive(132, 276)
    | (_inclusive(294, 347) - {321})
    | {390, 391, 395, 396, 397, 458, 487, 511, 512, 516, 517, 518, 519}
)
ANIMAL_CONTEXT_CLASS_SET = _inclusive(67, 131)
TONAL_CONTEXT_CLASS_SET = {495, 496, 497}
CONTEXT_CLASS_SET = ANIMAL_CONTEXT_CLASS_SET | TONAL_CONTEXT_CLASS_SET
EXCLUDED_DIRECT_CLASS_SET = {494} | _inclusive(500, 506) | {520}

ANIMAL_FOREGROUND_CLASS_SET = {
    69, 70, 71, 72, 73, 76, 77, 78, 79, 80, 82, 83, 84, 85, 86, 88,
    89, 90, 91, 92, 94, 95, 96, 97, 98, 99, 100, 101, 102, 104, 105,
    112, 113, 114, 115, 127, 128, 129, 130,
}
ANIMAL_DIFFUSE_CLASS_SET = ANIMAL_CONTEXT_CLASS_SET - ANIMAL_FOREGROUND_CLASS_SET

YAMNET_CLASS_GROUPS = {
    'CONSERVER_PFX': PFX_CLASS_SET,
    'RETIRER_HUMAIN': HUMAN_CLASS_SET,
    'RETIRER_AMBIANCE': AMBIENCE_CLASS_SET,
    'RETIRER_HORS_PFX': OUTSIDE_PFX_CLASS_SET,
    'TRAITER_CONTEXTE': CONTEXT_CLASS_SET,
    'EXCLURE_MASQUE_DIRECT': EXCLUDED_DIRECT_CLASS_SET,
}
EXPECTED_GROUP_COUNTS = {
    'CONSERVER_PFX': 146,
    'RETIRER_HUMAIN': 61,
    'RETIRER_AMBIANCE': 26,
    'RETIRER_HORS_PFX': 211,
    'TRAITER_CONTEXTE': 68,
    'EXCLURE_MASQUE_DIRECT': 9,
}

assigned = set()
for group_name, group_set in YAMNET_CLASS_GROUPS.items():
    overlap = assigned & group_set
    if overlap:
        raise RuntimeError(f"Indices YAMNet présents dans plusieurs groupes : {sorted(overlap)}")
    assigned |= group_set
    if len(group_set) != EXPECTED_GROUP_COUNTS[group_name]:
        raise RuntimeError(
            f"Compte invalide pour {group_name}: {len(group_set)} au lieu de "
            f"{EXPECTED_GROUP_COUNTS[group_name]}."
        )
if assigned != set(range(521)):
    missing = sorted(set(range(521)) - assigned)
    extra = sorted(assigned - set(range(521)))
    raise RuntimeError(f"Partition YAMNet incomplète — absents={missing}, hors plage={extra}")

for group_name, group_set in YAMNET_CLASS_GROUPS.items():
    print(f"   • {group_name}: {len(group_set)} classes")
print("   ✅ Audit taxonomie : 521/521 indices exacts, aucun chevauchement.")

HUMAN_CLASS_IDX = sorted(HUMAN_CLASS_SET)
PFX_CLASS_IDX = sorted(PFX_CLASS_SET)
AMBIENCE_CLASS_IDX = sorted(AMBIENCE_CLASS_SET)
OUTSIDE_PFX_CLASS_IDX = sorted(OUTSIDE_PFX_CLASS_SET)
ANIMAL_CONTEXT_CLASS_IDX = sorted(ANIMAL_CONTEXT_CLASS_SET)
ANIMAL_FOREGROUND_CLASS_IDX = sorted(ANIMAL_FOREGROUND_CLASS_SET)
ANIMAL_DIFFUSE_CLASS_IDX = sorted(ANIMAL_DIFFUSE_CLASS_SET)
TONAL_CONTEXT_CLASS_IDX = sorted(TONAL_CONTEXT_CLASS_SET)
EXCLUDED_DIRECT_CLASS_IDX = sorted(EXCLUDED_DIRECT_CLASS_SET)

SOFT_HUMAN_CLASS_IDX = [12, 23, 36]  # Whispering, Sigh, Breathing
assert set(SOFT_HUMAN_CLASS_IDX) <= HUMAN_CLASS_SET
RESPIRATORY_CLASS_IDX = [12, 23, 36, 39, 40, 41, 45]
CRYING_CLASS_IDX = [19, 20, 21, 22]
assert set(CRYING_CLASS_IDX) <= HUMAN_CLASS_SET
RESIDUAL_ORAL_CLASS_IDX = [49, 50, 51]
RESIDUAL_SPEECH_CLASS_IDX = [0, 1, 2, 3, 4]
assert set(RESPIRATORY_CLASS_IDX + RESIDUAL_ORAL_CLASS_IDX + RESIDUAL_SPEECH_CLASS_IDX) <= HUMAN_CLASS_SET
# Sous-ensembles additionnels, sans changer la partition des 521 classes.
MOUTH_CLASS_IDX = [49, 50, 51]  # Indices buccaux explicites.
MOUTH_TRANSIENT_CLASS_IDX = [428, 461, 485]  # Ambigus : contexte humain obligatoire.
MOUTH_CONTEXT_CLASS_IDX = [0, 1, 2, 3, 4, 12, 23, 36, 49, 50, 51]
BODYTALK_CLASS_IDX = [372, 470, 473, 481]  # Proxies, pas un détecteur spécialisé de vêtements.
assert set(MOUTH_CLASS_IDX) <= HUMAN_CLASS_SET
assert set(MOUTH_TRANSIENT_CLASS_IDX + BODYTALK_CLASS_IDX) <= PFX_CLASS_SET
assert 48 not in BODYTALK_CLASS_IDX  # Pas de boost explicite des pas.
IMPACT_GUARD_CLASS_IDX = [48, 353, 354, 413, 454, 460, 461]
YAMNET_FRAME_HOP_S = 0.48
YAMNET_MASK_BLOCK_S = 30.0
YAMNET_THRESHOLDS = {
    'human': (0.18, 0.45),
    'soft_human': (0.10, 0.30),  # Seuils expérimentaux, pas probabilités calibrées.
    'mouth': (0.08, 0.25),
    'mouth_transient': (0.10, 0.35),
    'human_presence': (0.06, 0.20),
    'bodytalk': (0.15, 0.40),
    'pfx': (0.12, 0.35),
    'ambience': (0.12, 0.38),
    'outside_pfx': (0.15, 0.42),
    'context': (0.14, 0.40),
}

def _max_class_score(scores, indices):
    if scores.shape[0] == 0 or not indices:
        return np.zeros(scores.shape[0], dtype=np.float32)
    return np.max(scores[:, indices], axis=1).astype(np.float32)

def _soft_threshold(values, low, high):
    if not 0.0 <= low < high <= 1.0:
        raise ValueError(f"Seuil YAMNet invalide : {low}, {high}")
    return np.clip((np.asarray(values, dtype=np.float32) - low) / (high - low), 0.0, 1.0)

def _moving_mean_frames(values, window_s):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values.copy()
    width = max(1, int(round(window_s / YAMNET_FRAME_HOP_S)))
    if width == 1:
        return values.copy()
    left = width // 2
    right = width - 1 - left
    padded = np.pad(values, (left, right), mode='edge')
    kernel = np.ones(width, dtype=np.float32) / width
    return np.convolve(padded, kernel, mode='valid').astype(np.float32)

def _attack_release_frames(values, attack_s, release_s):
    values = np.clip(np.asarray(values, dtype=np.float32), 0.0, 1.0)
    if values.size == 0:
        return values.copy()
    attack_alpha = np.exp(-YAMNET_FRAME_HOP_S / max(float(attack_s), 1e-3))
    release_alpha = np.exp(-YAMNET_FRAME_HOP_S / max(float(release_s), 1e-3))
    envelope = np.empty_like(values)
    previous = 0.0
    for i, target in enumerate(values):
        alpha = attack_alpha if target > previous else release_alpha
        previous = alpha * previous + (1.0 - alpha) * float(target)
        envelope[i] = previous
    return np.clip(envelope, 0.0, 1.0).astype(np.float32)

def _neighbor_peak_frames(values, radius=1):
    """Contexte +/- un hop (0,48 s), hors ligne; jamais le clip entier."""
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values.copy()
    padded = np.pad(values, (radius, radius), mode='constant')
    return np.maximum.reduce([
        padded[offset:offset + len(values)] for offset in range(2 * radius + 1)
    ]).astype(np.float32)

def _respiratory_frames(scores, reinforced):
    if not reinforced:
        return _attack_release_frames(
            _soft_threshold(_max_class_score(scores, SOFT_HUMAN_CLASS_IDX), 0.10, 0.30),
            attack_s=0.08, release_s=0.60,
        )
    activity = np.sqrt(_soft_threshold(
        _max_class_score(scores, RESPIRATORY_CLASS_IDX), 0.06, 0.18
    ))
    return _attack_release_frames(
        _neighbor_peak_frames(activity), attack_s=0.05, release_s=0.55
    )

def _residual_frames(scores):
    # Seulement des classes humaines explicites. Pas de retrait additionnel
    # des impacts/clics génériques, de la nature ou des objets.
    breaths = np.sqrt(_soft_threshold(
        _max_class_score(scores, RESPIRATORY_CLASS_IDX), 0.08, 0.25
    ))
    oral = _soft_threshold(_max_class_score(scores, RESIDUAL_ORAL_CLASS_IDX), 0.10, 0.30)
    speech = _soft_threshold(_max_class_score(scores, RESIDUAL_SPEECH_CLASS_IDX), 0.10, 0.35)
    crying = _crying_activity(scores)
    activity = np.maximum.reduce([breaths, oral, speech, crying])
    residual = _attack_release_frames(
        _neighbor_peak_frames(activity), attack_s=0.05, release_s=0.45
    )
    guard = _attack_release_frames(
        _neighbor_peak_frames(_soft_threshold(
            _max_class_score(scores, HUMAN_CLASS_IDX), 0.06, 0.20
        )), attack_s=0.04, release_s=0.60
    )
    return {'residual': residual, 'human_guard': guard}

def _crying_activity(scores):
    if DUCK_DEPTH_PLEURS <= 0:
        return np.zeros(scores.shape[0], dtype=np.float32)
    return np.sqrt(_soft_threshold(
        _max_class_score(scores, CRYING_CLASS_IDX), 0.06, 0.20
    ))

def _crying_frames(scores):
    return _attack_release_frames(
        _neighbor_peak_frames(_crying_activity(scores)), attack_s=0.05, release_s=0.65
    )

def _residual_scores_with_gain(wave, base_scores, gain_db):
    """Max des scores aux deux niveaux, sans cumuler deux retraits.
    
    YAMNet : fenêtres effectives 15 600 samples à 16 kHz et hop 7 680.
    Chaque appel amplifié doit produire exactement une frame.
    """
    gain_db = float(gain_db)
    if not np.isfinite(gain_db) or not 0.0 <= gain_db <= 24.0:
        raise ValueError("Gain d'analyse résiduelle hors plage 0–24 dB.")
    if gain_db == 0.0 or len(base_scores) == 0:
        return base_scores
    combined = base_scores.copy()
    requested = 10.0 ** (gain_db / 20.0)
    for i in range(len(base_scores)):
        segment = wave[i * 7680:i * 7680 + 15600]
        peak = float(np.max(np.abs(segment))) if segment.size else 0.0
        if peak == 0.0:
            continue
        # Pas de réduction du niveau initial si la crête est déjà > 0,90.
        gain = max(1.0, min(requested, 0.90 / peak))
        if gain <= 1.0:
            continue
        boosted = np.pad(segment, (0, 15600 - len(segment))) * gain
        extra, _embeddings, _spectrogram = yamnet_model(boosted.astype(np.float32))
        extra = extra.numpy().astype(np.float32)
        if extra.shape != (1, 521):
            raise RuntimeError(f"Résultat YAMNet amplifié inattendu : {extra.shape}")
        combined[i] = np.maximum(base_scores[i], extra[0])
    return combined

def _analyze_residual_audio(audio, sr):
    mono = np.mean(audio, axis=1).astype(np.float32)
    if mono.size == 0 or not np.any(mono):
        return {'frames': _residual_frames(np.zeros((0, 521), dtype=np.float32))}
    if not np.all(np.isfinite(mono)):
        raise ValueError("Mix non fini avant l'analyse des résidus YAMNet.")
    divisor = int(np.gcd(int(sr), 16000))
    wave = signal.resample_poly(mono, up=16000 // divisor, down=int(sr) // divisor).astype(np.float32)
    peak = float(np.max(np.abs(wave)))
    if peak > 1.0:
        wave /= peak
    # Première observation au niveau d'origine; elle reste conservée dans le maximum.
    scores, _embeddings, _spectrogram = yamnet_model(wave)
    scores = scores.numpy().astype(np.float32)
    combined = _residual_scores_with_gain(wave, scores, GAIN_ANALYSE_RESIDUS_DB)
    print(f"      Analyse résidus : original + gain local maximal {GAIN_ANALYSE_RESIDUS_DB:g} dB (détecteur seulement)")
    return {'frames': _residual_frames(combined)}

def _mouth_bodytalk_frames(scores):
    # Les clics/impacts génériques seuls ne prouvent PAS un bruit de bouche.
    oral = _soft_threshold(_max_class_score(scores, MOUTH_CLASS_IDX), *YAMNET_THRESHOLDS['mouth'])
    context = _soft_threshold(
        _max_class_score(scores, MOUTH_CONTEXT_CLASS_IDX), *YAMNET_THRESHOLDS['human_presence']
    )
    transient = _soft_threshold(
        _max_class_score(scores, MOUTH_TRANSIENT_CLASS_IDX), *YAMNET_THRESHOLDS['mouth_transient']
    )
    mouth = np.maximum(oral, transient * _neighbor_peak_frames(context))
    # Une frame de marge couvre mieux les amorces de phrase / smacks courts.
    # Coût assumé : atténuation possible du foley dans cette fenêtre.
    mouth = _attack_release_frames(_neighbor_peak_frames(mouth), attack_s=0.04, release_s=0.25)
    human_presence = _soft_threshold(
        _max_class_score(scores, HUMAN_CLASS_IDX), *YAMNET_THRESHOLDS['human_presence']
    )
    human_guard = _attack_release_frames(
        _neighbor_peak_frames(human_presence), attack_s=0.04, release_s=0.60
    )
    bodytalk = _attack_release_frames(
        _soft_threshold(_max_class_score(scores, BODYTALK_CLASS_IDX), *YAMNET_THRESHOLDS['bodytalk']),
        attack_s=0.12, release_s=0.40
    )
    return {'mouth': mouth, 'bodytalk': bodytalk, 'human_guard': human_guard}

def yamnet_mask_block(yamnet_data, mask_name, start, end, target_sr):
    if yamnet_data is None or end <= start:
        return np.zeros(max(0, end - start), dtype=np.float32)
    frame_values = np.asarray(yamnet_data.get('frames', {}).get(mask_name, []), dtype=np.float32)
    if frame_values.size == 0:
        return np.zeros(end - start, dtype=np.float32)
    frame_times = np.arange(frame_values.size, dtype=np.float64) * YAMNET_FRAME_HOP_S + YAMNET_FRAME_HOP_S
    sample_times = np.arange(start, end, dtype=np.float64) / float(target_sr)
    return np.interp(
        sample_times, frame_times, frame_values,
        left=float(frame_values[0]), right=float(frame_values[-1]),
    ).astype(np.float32)

def analyze_yamnet(filepath):
    audio, sr = sf.read(filepath, always_2d=True)
    mono = np.mean(audio, axis=1).astype(np.float32)

    if int(sr) == 48000:
        mono_16k = signal.resample_poly(mono, up=1, down=3).astype(np.float32)
    else:
        divisor = int(np.gcd(int(sr), 16000))
        mono_16k = signal.resample_poly(
            mono, up=16000 // divisor, down=int(sr) // divisor
        ).astype(np.float32)

    peak = float(np.max(np.abs(mono_16k))) if mono_16k.size else 0.0
    if peak > 1.0:
        mono_16k = mono_16k / peak

    if mono_16k.size == 0:
        return {
            'frames': {name: np.zeros(0, dtype=np.float32) for name in ('human', 'soft_human', 'mouth', 'bodytalk', 'human_guard', 'ambience', 'outside_pfx', 'pfx')},
            'source_sr': int(sr), 'source_length': 0, 'top_classes': [],
        }

    scores, _embeddings, _spectrogram = yamnet_model(mono_16k)
    scores = scores.numpy().astype(np.float32)
    n_frames = scores.shape[0]
    if n_frames == 0:
        return {
            'frames': {name: np.zeros(0, dtype=np.float32) for name in ('human', 'soft_human', 'mouth', 'bodytalk', 'human_guard', 'ambience', 'outside_pfx', 'pfx')},
            'source_sr': int(sr), 'source_length': len(mono), 'top_classes': [],
        }

    human_raw = _max_class_score(scores, HUMAN_CLASS_IDX)
    soft_human_raw = _max_class_score(scores, SOFT_HUMAN_CLASS_IDX)
    pfx_raw = _max_class_score(scores, PFX_CLASS_IDX)
    ambience_raw = _max_class_score(scores, AMBIENCE_CLASS_IDX)
    outside_raw = _max_class_score(scores, OUTSIDE_PFX_CLASS_IDX)
    animal_raw = _max_class_score(scores, ANIMAL_CONTEXT_CLASS_IDX)
    animal_foreground_raw = _max_class_score(scores, ANIMAL_FOREGROUND_CLASS_IDX)
    animal_diffuse_raw = _max_class_score(scores, ANIMAL_DIFFUSE_CLASS_IDX)
    tonal_raw = _max_class_score(scores, TONAL_CONTEXT_CLASS_IDX)

    human_base = _soft_threshold(human_raw, *YAMNET_THRESHOLDS['human'])
    soft_human_base = _soft_threshold(soft_human_raw, *YAMNET_THRESHOLDS['soft_human'])
    pfx_base = _soft_threshold(pfx_raw, *YAMNET_THRESHOLDS['pfx'])
    ambience_base = _soft_threshold(ambience_raw, *YAMNET_THRESHOLDS['ambience'])
    outside_base = _soft_threshold(outside_raw, *YAMNET_THRESHOLDS['outside_pfx'])

    animal_activity = _soft_threshold(animal_raw, *YAMNET_THRESHOLDS['context'])
    animal_foreground = _soft_threshold(animal_foreground_raw, *YAMNET_THRESHOLDS['context'])
    animal_diffuse = _soft_threshold(animal_diffuse_raw, *YAMNET_THRESHOLDS['context'])
    animal_persistence = _soft_threshold(_moving_mean_frames(animal_activity, 4.0), 0.30, 0.70)
    diffuse_persistence = _soft_threshold(_moving_mean_frames(animal_diffuse, 2.5), 0.20, 0.60)
    animal_remove = np.maximum(
        animal_activity * animal_persistence,
        animal_diffuse * diffuse_persistence,
    )
    animal_keep = animal_foreground * (1.0 - animal_persistence)

    tonal_activity = _soft_threshold(tonal_raw, *YAMNET_THRESHOLDS['context'])
    tonal_persistence = _soft_threshold(_moving_mean_frames(tonal_activity, 2.5), 0.35, 0.75)
    tonal_remove = tonal_activity * tonal_persistence
    tonal_keep = tonal_activity * (1.0 - tonal_persistence)

    frame_masks = {
        'crying': _crying_frames(scores),
        'soft_human': _respiratory_frames(scores, PROFIL_RESPIRATOIRE_RENFORCE),
        'human': _attack_release_frames(human_base, attack_s=0.12, release_s=0.80),
        'ambience': _attack_release_frames(
            np.maximum(ambience_base, animal_remove), attack_s=0.45, release_s=3.50
        ),
        'outside_pfx': _attack_release_frames(
            np.maximum(outside_base, tonal_remove), attack_s=0.20, release_s=1.20
        ),
        'pfx': _attack_release_frames(
            np.maximum.reduce([pfx_base, animal_keep, tonal_keep]), attack_s=0.05, release_s=0.60
        ),
    }

    frame_masks.update(_mouth_bodytalk_frames(scores))
    frame_masks['impact_guard'] = _attack_release_frames(
        _neighbor_peak_frames(_soft_threshold(_max_class_score(scores, IMPACT_GUARD_CLASS_IDX), .08, .25)),
        attack_s=.04, release_s=.30)

    crying_peak = _max_class_score(scores, CRYING_CLASS_IDX)
    print(f"      Pleurs — score max : {float(np.max(crying_peak)):.3f} | masque actif : {100.0 * np.mean(frame_masks['crying'] > 0.10):.1f}% | retrait : {DUCK_DEPTH_PLEURS}%")

    per_class_peak = np.max(scores, axis=0)
    print('      Souffles — pics : ' + ', '.join(
        f'{class_names[i]}={per_class_peak[i]:.3f}' for i in (RESPIRATORY_CLASS_IDX if PROFIL_RESPIRATOIRE_RENFORCE else SOFT_HUMAN_CLASS_IDX)
    ))
    print(f"      Souffles — masque actif : {100.0 * np.mean(frame_masks['soft_human'] > 0.10):.1f}% | profondeur : {DUCK_DEPTH_SOUFFLES}%")
    print('      Bouche/bodytalk — pics : ' + ', '.join(
        f'{class_names[i]}={per_class_peak[i]:.3f}'
        for i in MOUTH_CLASS_IDX + MOUTH_TRANSIENT_CLASS_IDX + BODYTALK_CLASS_IDX
    ))
    print(
        f"      Bouche : {100.0 * np.mean(frame_masks['mouth'] > 0.10):.1f}% actif | "
        f"retrait {DUCK_DEPTH_BOUCHE}% | bodytalk demandé +{GAIN_BODYTALK_DB:.1f} dB"
    )
    top_indices = np.argsort(per_class_peak)[-8:][::-1]
    top_classes = [(int(i), class_names[i], float(per_class_peak[i])) for i in top_indices]
    active = {name: 100.0 * float(np.mean(mask > 0.10)) for name, mask in frame_masks.items()}
    print(
        f"      Masques actifs — humain {active['human']:.1f}% | ambiance {active['ambience']:.1f}% | "
        f"hors-PFX {active['outside_pfx']:.1f}% | protection PFX {active['pfx']:.1f}%"
    )
    print("      Top YAMNet : " + ", ".join(f"#{index} {name}={score:.2f}" for index, name, score in top_classes[:5]))

    return {
        'frames': frame_masks,
        'source_sr': int(sr),
        'source_length': len(mono),
        'top_classes': top_classes,
    }

print("\n✅ Détecteur YAMNet PFX prêt.")


In [ ]:
# 6. TEST V3 — scène indépendante, aucun décalage ni regroupement
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]


def get_sample_rate(filepath):
    try:
        _, sr = sf.read(filepath, frames=1)
        return sr
    except: return 48000


def inject_timecode(original_path, processed_path, final_output_path):
    original_sr = get_sample_rate(original_path)
    try:
        probe_cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format_tags=time_reference', '-of', 'default=nw=1:nk=1', original_path]
        time_ref = subprocess.check_output(probe_cmd).decode('utf-8').strip()
    except Exception: time_ref = ""

    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-i', processed_path, '-i', original_path, '-map', '0:a', '-map_metadata', '1', '-write_bext', '1', '-metadata:g', 'umid=']
    if time_ref: cmd.extend(['-metadata:g', f'time_reference={time_ref}'])
    cmd.extend(['-ar', str(original_sr), '-c:a', 'pcm_f32le', final_output_path])
    subprocess.run(cmd, check=True)
    assign_render_umid(final_output_path, forbidden=[read_wave_umid(original_path)])
    ensure_export_extensible(final_output_path)


def prep_denoise_batch(files_list, yamnet_cache):
    print(f"\n{'='*50}\n🧹 ÉTAPE 1 : PRÉ-NETTOYAGE PAR SCÈNE ({NIVEAU_DENOISE_POURCENTAGE}%, {'adaptatif' if DENOISE_ADAPTATIF else 'stationnaire'})\n{'='*50}")

    prop_decrease_val = NIVEAU_DENOISE_POURCENTAGE / 100.0
    stationary_flag = not DENOISE_ADAPTATIF

    for f in files_list:
        in_path = os.path.join(FOLDER_MERGED, f)
        out_path = os.path.join(FOLDER_DENOISED, f)

        print(f"   🚿 Pré-nettoyage ({NIVEAU_DENOISE_POURCENTAGE}%) : {f}")
        audio, sr = sf.read(in_path, always_2d=True)
        reduced_audio = np.zeros_like(audio)
        for i in range(audio.shape[1]):
            reduced_channel = nr.reduce_noise(
                y=audio[:, i], sr=sr, stationary=stationary_flag, prop_decrease=prop_decrease_val,
                n_fft=8192, win_length=8192, hop_length=512,
                time_mask_smooth_ms=150, freq_mask_smooth_hz=500
            )
            if np.isnan(reduced_channel).any(): reduced_channel = audio[:, i]
            reduced_audio[:, i] = reduced_channel

        base, _ = os.path.splitext(f)
        yamnet_data = yamnet_cache.get(base)
        if yamnet_data is not None and PROTECTION_PFX > 0:
            block_size = max(1, int(round(sr * YAMNET_MASK_BLOCK_S)))
            active_samples = 0
            protection_depth = min(PROTECTION_PFX / 100.0, 0.90)
            for start in range(0, audio.shape[0], block_size):
                end = min(start + block_size, audio.shape[0])
                pfx_mask = yamnet_mask_block(yamnet_data, 'pfx', start, end, sr)
                restore = np.clip(pfx_mask * protection_depth, 0.0, 0.90)
                active_samples += int(np.count_nonzero(restore > 0.05))
                restore_col = restore.reshape(-1, 1)
                reduced_audio[start:end] = (
                    audio[start:end] * restore_col
                    + reduced_audio[start:end] * (1.0 - restore_col)
                )
            pct = 100.0 * active_samples / max(1, audio.shape[0])
            print(f"      🛡️ Protection PFX : {pct:.1f}% du fichier partiellement restauré")

        sf.write(out_path, reduced_audio, sr, subtype='FLOAT')


def _resample_to_rate(data, source_sr, target_sr, axis=0):
    source_sr = int(source_sr)
    target_sr = int(target_sr)
    if source_sr <= 0 or target_sr <= 0:
        raise ValueError(f"Fréquence d'échantillonnage invalide : {source_sr} -> {target_sr}")
    if source_sr == target_sr:
        return np.asarray(data)
    divisor = int(np.gcd(source_sr, target_sr))
    return signal.resample_poly(
        data, up=target_sr // divisor, down=source_sr // divisor, axis=axis
    )


"""Test V3 delivery contract, also embedded in the standalone Colab notebook."""
import hashlib
import json
import re
import struct
import uuid
from pathlib import Path

DELIVERY_MANIFEST = 'PFX_delivery.json'
RENDER_UMID_PREFIX = bytes.fromhex('060a2b340101010501010f1013000000')


def wave_chunk_extent(path, wanted):
    """Locate one RIFF chunk with bounds checks; no audio decoding or mutation."""
    with Path(path).open('rb') as handle:
        header = handle.read(12)
        if len(header) != 12 or header[:4] != b'RIFF' or header[8:] != b'WAVE':
            raise ValueError(f'WAV RIFF invalide : {Path(path).name}')
        end = struct.unpack_from('<I', header, 4)[0] + 8
        handle.seek(0, 2)
        if end > handle.tell():
            raise ValueError('WAV tronqué.')
        handle.seek(12)
        matches = []
        while handle.tell() + 8 <= end:
            tag, size = struct.unpack('<4sI', handle.read(8))
            start = handle.tell()
            if start + size > end:
                raise ValueError('Chunk WAV tronqué.')
            if tag == wanted:
                matches.append((start, size))
            handle.seek(start + size + size % 2)
        if len(matches) != 1:
            raise ValueError(f'Chunk {wanted!r} absent ou dupliqué : {Path(path).name}')
        return matches[0]


def read_wave_umid(path):
    offset, size = wave_chunk_extent(path, b'bext')
    if size < 412:
        raise ValueError(f'UMID BWF absent : {Path(path).name}')
    with Path(path).open('rb') as handle:
        handle.seek(offset + 348)
        umid = handle.read(64)
    if not any(umid[:32]):
        raise ValueError(f'UMID BWF vide : {Path(path).name}')
    return umid.hex()


def wave_audio_sha256(path):
    offset, size = wave_chunk_extent(path, b'data')
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        handle.seek(offset)
        while size:
            block = handle.read(min(size, 1024 * 1024))
            if not block:
                raise ValueError('Audio tronqué.')
            digest.update(block)
            size -= len(block)
    return digest.hexdigest()


def assign_render_umid(path, forbidden=()):
    """Only for newly rendered staging WAVs. Fresh native-compatible basic UMID.

    Keep the observed native label/zero instance and use a UUID4 material number.
    Write bytes directly AFTER FFmpeg; never stringify via signed strtoll in FFmpeg.
    """
    forbidden_basic = {value[:64].lower() for value in forbidden}
    for _ in range(10):
        umid = RENDER_UMID_PREFIX + uuid.uuid4().bytes + bytes(32)
        if umid[:32].hex() not in forbidden_basic:
            break
    else:
        raise RuntimeError('Impossible de générer un UMID distinct; livraison refusée.')
    offset, size = wave_chunk_extent(path, b'bext')
    if size < 412:
        raise ValueError('BWF trop court pour écrire un UMID.')
    before = wave_audio_sha256(path)
    with Path(path).open('r+b') as handle:
        handle.seek(offset + 348)
        handle.write(umid)
    if read_wave_umid(path) != umid.hex() or wave_audio_sha256(path) != before:
        raise RuntimeError('Échec de vérification UMID/audio; livraison refusée.')
    return umid.hex()


def verify_unique_wave_umids(paths, forbidden=()):
    seen = {value[:64].lower(): 'source' for value in forbidden}
    for path in paths:
        basic = read_wave_umid(path)[:64]
        if basic in seen:
            raise ValueError(f'UMID dupliqué : {Path(path).name} et {seen[basic]}. '
                             'Import Pro Tools dangereux; livraison refusée. Refaire avec le backend corrigé.')
        seen[basic] = Path(path).name


def safe_audio_name(name):
    if (not isinstance(name, str) or not name or name in ('.', '..')
            or re.search(r'[<>:"/\\|?*\x00-\x1f]', name)
            or name.endswith((' ', '.'))
            or name.split('.')[0].upper() in {'CON', 'PRN', 'AUX', 'NUL',
                *(f'COM{i}' for i in range(1, 10)), *(f'LPT{i}' for i in range(1, 10))}
            or not name.lower().endswith('.wav')):
        raise ValueError(f'Nom WAV non compatible Windows : {name!r}')
    return name


def microphone_role(name):
    safe_audio_name(name)
    roles = [role for role in ('BOOM', 'LAV') if role.casefold() in name.casefold()]
    if len(roles) != 1:
        raise ValueError(f'Nom ambigu : un seul identifiant BOOM ou LAV requis : {name}')
    return roles[0]


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def inspect_delivery_wave(path, for_export=False):
    """Read RIFF geometry/BWF time reference, not audio; reject unsupported inputs early."""
    path = Path(path)
    chunks = {}
    with path.open('rb') as handle:
        size = path.stat().st_size
        header = handle.read(12)
        if len(header) != 12 or header[:4] != b'RIFF' or header[8:] != b'WAVE':
            raise ValueError(f'WAV RIFF requis (RF64 non pris en charge) : {path.name}')
        riff_end = struct.unpack_from('<I', header, 4)[0] + 8
        if riff_end > size:
            raise ValueError(f'WAV tronqué : {path.name}')
        while handle.tell() + 8 <= riff_end:
            tag, length = struct.unpack('<4sI', handle.read(8))
            offset = handle.tell()
            if offset + length > riff_end:
                raise ValueError(f'Chunk WAV tronqué : {path.name}')
            if tag in (b'fmt ', b'data', b'bext', b'fact'):
                if tag in chunks:
                    raise ValueError(f'Chunk WAV dupliqué : {path.name}')
                chunks[tag] = (length, handle.read(min(length, 412)) if tag != b'data' else b'')
            handle.seek(offset + length + length % 2)
    if not all(k in chunks for k in (b'fmt ', b'data', b'bext')):
        raise ValueError(f'WAV sans format, audio ou timecode BWF : {path.name}')
    fmt = chunks[b'fmt '][1]
    if len(fmt) < 16 or chunks[b'bext'][0] < 412:
        raise ValueError(f'Entête WAV/BWF invalide : {path.name}')
    code, channels, rate, byte_rate, align, bits = struct.unpack_from('<HHIIHH', fmt)
    if code == 0xfffe and len(fmt) >= 40:
        if fmt[28:40] != bytes.fromhex('00001000800000aa00389b71'):
            raise ValueError(f'Sous-format WAV inconnu : {path.name}')
        code = struct.unpack_from('<I', fmt, 24)[0]
    if (channels != 1 or rate != 48000 or code not in (1, 3)
            or bits not in ((16, 24, 32) if code == 1 else (32,))
            or align != bits // 8 or byte_rate != rate * align
            or chunks[b'data'][0] % align):
        raise ValueError(f'WAV mono PCM/float 48 kHz à échantillons complets requis : {path.name}')
    frames = chunks[b'data'][0] // align
    # Current pt_api template builder geometry; do not silently pad or truncate.
    if not 1024 <= frames <= 0xffffff:
        raise ValueError(f'Durée hors profil export PTX (1024–16777215 samples) : {path.name}')
    time_reference = struct.unpack_from('<Q', chunks[b'bext'][1], 338)[0]
    if (time_reference > 0xffffffff or not any(chunks[b'bext'][1][348:380])
            or struct.unpack_from('<H', chunks[b'bext'][1], 346)[0] not in (1, 2)):
        raise ValueError(f'Timecode hors profil UInt32 ou UMID BWF absent : {path.name}')
    if for_export:
        fact = chunks.get(b'fact', (0, b''))[1]
        if (len(fmt) < 40 or struct.unpack_from('<H', fmt)[0] != 0xfffe
                or code != 3 or bits != 32 or struct.unpack_from('<H', fmt, 16)[0] < 22
                or struct.unpack_from('<H', fmt, 18)[0] != 32
                or len(fact) < 4 or struct.unpack_from('<I', fact)[0] != frames):
            raise ValueError(f'Export PTX : float32 extensible/fact correct requis : {path.name}')
    return {'frames': frames, 'sample_rate': rate, 'time_reference': time_reference}


def validate_delivery_manifest(manifest):
    if (not isinstance(manifest, dict) or manifest.get('protocol') not in (1, 2)
            or manifest.get('status') != 'complete'):
        raise ValueError('Traitement incomplet ou manifeste incompatible; téléchargement refusé.')
    run_id = manifest.get('run_id', '')
    if not isinstance(run_id, str) or not re.fullmatch(r'[A-Za-z0-9_-]+', run_id):
        raise ValueError('Identifiant de traitement invalide.')
    if manifest.get('run_folder') != 'run_' + run_id:
        raise ValueError('Dossier de traitement invalide.')
    outputs = manifest.get('outputs')
    if not isinstance(outputs, list) or not outputs:
        raise ValueError('Manifeste sans WAV.')
    names = set()
    identities = set()
    source_identities = {e['umid'][:64] for e in manifest.get('inputs', []) if 'umid' in e}
    for item in outputs:
        name = safe_audio_name(item['name'])
        if name.casefold() in names:
            raise ValueError('WAV dupliqué dans le manifeste.')
        names.add(name.casefold())
        if item.get('role') != microphone_role(name):
            raise ValueError('Rôle BOOM/LAV incohérent dans le manifeste.')
        if not re.fullmatch(r'[0-9a-f]{64}', item.get('sha256', '')):
            raise ValueError('Empreinte WAV invalide.')
        if (type(item.get('frames')) is not int or not 1024 <= item['frames'] <= 0xffffff
                or item.get('sample_rate') != 48000
                or type(item.get('time_reference')) is not int or item['time_reference'] < 0):
            raise ValueError('Géométrie WAV invalide dans le manifeste.')
        if manifest['protocol'] == 2:
            umid = item.get('umid', '')
            if (not isinstance(umid, str) or not re.fullmatch(r'[0-9a-f]{128}', umid)
                    or not umid.startswith(RENDER_UMID_PREFIX.hex()) or umid[64:] != '0' * 64):
                raise ValueError('UMID de rendu invalide dans le manifeste.')
            if umid[:64] in identities or umid[:64] in source_identities:
                raise ValueError('UMID dupliqué ou réutilisé depuis le brut; livraison refusée.')
            identities.add(umid[:64])
    return outputs


def verify_delivery_file(path, entry):
    if sha256_file(path) != entry['sha256']:
        raise ValueError(f'Empreinte incorrecte, WAV incomplet ou mauvais lot : {entry["name"]}')
    info = inspect_delivery_wave(path, for_export=True)
    if any(info[k] != entry[k] for k in ('frames', 'sample_rate', 'time_reference')):
        raise ValueError(f'Durée/timecode incorrect : {entry["name"]}')
    if 'umid' in entry and read_wave_umid(path) != entry['umid']:
        raise ValueError(f'UMID différent du manifeste : {entry["name"]}')


def ensure_export_extensible(path):
    """Normalize a plain IEEE-float fmt header only; never touch data/BWF payloads.

    Some FFmpeg versions emit fmt=3 (18 bytes) for mono, others extensible (40).
    pt_api requires the latter. Only our newly rendered staging WAV is rewritten.
    """
    path = Path(path)
    before = inspect_delivery_wave(path)
    data = path.read_bytes()
    pos = 12
    while pos + 8 <= len(data):
        tag, size = struct.unpack_from('<4sI', data, pos)
        end = pos + 8 + size + size % 2
        if tag == b'fmt ':
            fmt = data[pos + 8:pos + 8 + size]
            code = struct.unpack_from('<H', fmt)[0]
            if code == 0xfffe:
                inspect_delivery_wave(path, for_export=True)
                return
            if code != 3 or struct.unpack_from('<H', fmt, 14)[0] != 32:
                raise ValueError('Seul un rendu float32 peut être normalisé pour PTX.')
            replacement = struct.pack('<HHIIHHHHI', 0xfffe, 1, 48000, 192000, 4, 32, 22, 32, 4)
            replacement += bytes.fromhex('0300000000001000800000aa00389b71')
            normalized = bytearray(data[:pos] + b'fmt ' + struct.pack('<I', 40) + replacement + data[end:])
            struct.pack_into('<I', normalized, 4, len(normalized) - 8)
            pending = path.with_name(path.name + '.extensible.part')
            pending.write_bytes(normalized)
            if inspect_delivery_wave(pending, for_export=True) != before:
                raise ValueError('Géométrie modifiée pendant la normalisation WAV.')
            pending.replace(path)
            return
        pos = end
    raise ValueError('Chunk fmt absent.')


def write_delivery_manifest(path, manifest):
    path = Path(path)
    pending = path.with_name(path.name + '.part')
    pending.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    pending.replace(path)

"""Test V3 finishing; embedded after YAMNet helpers in the Colab notebook."""


def finish_v3(stem_path, output_path):
    audio, sr = sf.read(stem_path, dtype='float32', always_2d=True)
    audio = validate_bandit_audio(audio, sr)
    # Both observations see the SAME unattenuated SFX, never an already ducked mix.
    post_masks = analyze_yamnet(stem_path)
    residual = _analyze_residual_audio(audio, sr) if DUCK_DEPTH_RESIDUS > 0 else {'frames': {}}
    result = apply_v3_masks(audio, sr, post_masks, residual)
    if not np.isfinite(result).all():
        raise RuntimeError('Finition V3 non finie; aucune publication.')
    sf.write(output_path, result, sr, subtype='FLOAT')


def apply_v3_masks(audio, sr, masks, residual):
    """One maximum attenuation, then guarded local gain. No isolated cloth stem claim."""
    from scipy.ndimage import maximum_filter1d, uniform_filter1d
    result = audio.copy()
    n = len(result)
    weight = np.zeros(n, np.float32)
    block_size = max(1, int(round(sr * YAMNET_MASK_BLOCK_S)))
    depths = {'human': DUCK_DEPTH_HUMAIN / 100.0,
              'soft_human': DUCK_DEPTH_SOUFFLES / 100.0,
              'crying': DUCK_DEPTH_PLEURS / 100.0,
              'mouth': DUCK_DEPTH_BOUCHE / 100.0,
              'ambience': DUCK_DEPTH_AMBIANCE / 100.0,
              'outside_pfx': DUCK_DEPTH_HORS_PFX / 100.0}
    for start in range(0, n, block_size):
        end = min(n, start + block_size)
        values = {k: yamnet_mask_block(masks, k, start, end, sr)
                  for k in (*depths, 'pfx', 'bodytalk', 'human_guard', 'impact_guard')}
        res = yamnet_mask_block(residual, 'residual', start, end, sr)
        guard = yamnet_mask_block(residual, 'human_guard', start, end, sr)
        veto = np.maximum.reduce([values[k] for k in (*depths, 'human_guard', 'impact_guard')]
                                 + [res, guard])
        weight[start:end] = values['bodytalk'] * np.square(np.clip(1 - veto, 0, 1))
        pfx = values['pfx'] * PROTECTION_PFX / 100.0
        values['human'] *= 1 - .35 * pfx
        values['ambience'] *= 1 - .85 * pfx
        values['outside_pfx'] *= 1 - .55 * pfx
        reduction = np.maximum.reduce([values[k] * depths[k] for k in depths]
                                      + [res * np.clip(DUCK_DEPTH_RESIDUS / 100.0, 0, .95)])
        result[start:end] *= np.clip(1 - reduction, .03, 1).reshape(-1, 1)
    # Local look-ahead peak margin, smoothed conservatively: no unrelated global veto.
    radius = max(1, int(round(.25 * sr)))
    local_peak = maximum_filter1d(np.max(np.abs(result), axis=1), size=2 * radius + 1, mode='nearest')
    headroom = np.maximum(0, 20 * np.log10(.98 / np.maximum(local_peak, 1e-12)))
    requested = np.clip(float(GAIN_BODYTALK_DB), 0, 4) * weight
    available = np.minimum(headroom, 4.0)
    available = np.minimum(available, uniform_filter1d(available, size=2 * radius + 1, mode='nearest'))
    gain_db = np.minimum(requested, available)
    result *= np.power(10.0, gain_db / 20.0).astype(np.float32).reshape(-1, 1)
    print(f'      V3 : atténuation unique; bodytalk effectif max +{float(np.max(gain_db)):.2f} dB')
    return result


def preflight_v3_inputs():
    files = sorted([p for p in Path(FOLDER_IN).iterdir()
                    if p.is_file() and p.suffix.lower() == '.wav' and not p.name.startswith('._')],
                   key=lambda p: natural_sort_key(p.name))
    if not files:
        raise ValueError('Aucun WAV source.')
    seen = set()
    entries = []
    for path in files:
        role = microphone_role(path.name)
        output_name = path.stem + '_PFX_Ready.wav'
        safe_audio_name(output_name)
        if output_name.casefold() in seen:
            raise ValueError('Noms de sorties dupliqués sans tenir compte de la casse.')
        seen.add(output_name.casefold())
        info = inspect_delivery_wave(path)
        entries.append(dict(info, name=path.name, output_name=output_name,
                            role=role, sha256=sha256_file(path), umid=read_wave_umid(path)))
    counts = {role: sum(e['role'] == role for e in entries) for role in ('BOOM', 'LAV')}
    if all(counts.values()) and counts['BOOM'] != counts['LAV']:
        raise ValueError(f'Comptes BOOM/LAV différents : {counts}. Aucun appariement ni décalage appliqué.')
    return entries


In [ ]:
"""Standalone cell 7 source, executed in the Colab namespace."""
WORKFLOW_COMPLETE = False
pointer = Path(FOLDER_OUT) / DELIVERY_MANIFEST
if pointer.exists():
    previous = json.loads(pointer.read_text(encoding='utf-8'))
    if previous.get('status') == 'running' and not REMPLACER_RUN_INTERROMPU:
        raise RuntimeError('Un run est en cours ou interrompu. Vérifiez son arrêt, puis activez REMPLACER_RUN_INTERROMPU pour relancer.')
inputs_manifest = preflight_v3_inputs()  # Names, mono/48k, exact duration and BWF before models.
run_folder = 'run_' + RUN_ID
run_path = Path(FOLDER_OUT) / run_folder
run_path.mkdir(exist_ok=False)
manifest = {'protocol': 2, 'backend': 'Vtest_V3', 'revision': '2026-10-06-test-v3-umid',
            'run_id': RUN_ID, 'run_folder': run_folder, 'status': 'running',
            'inputs': inputs_manifest,
            'settings': {k: v for k, v in list(globals().items())
                         if k.startswith(('DUCK_DEPTH_', 'GAIN_', 'PROFIL_RESPIRATOIRE_'))
                         or k in ('NIVEAU_DENOISE_POURCENTAGE', 'DENOISE_ADAPTATIF', 'PROTECTION_PFX')},
            'upstream_commit': BANDIT_COMMIT, 'checkpoint_md5': BANDIT_CHECKPOINT_MD5,
            'alignment': False, 'scene_merge': False, 'outputs': []}
write_delivery_manifest(pointer, manifest)
try:
    # Snapshot each source without shifting it or borrowing another scene's context.
    for entry in inputs_manifest:
        local = Path(FOLDER_MERGED) / entry['name']
        shutil.copy2(Path(FOLDER_IN) / entry['name'], local)
        if sha256_file(local) != entry['sha256']:
            raise RuntimeError(f"Source modifiée pendant la copie : {entry['name']}")
    preflight_models()
    names = [entry['name'] for entry in inputs_manifest]
    raw_masks = {}
    for entry in inputs_manifest:
        name = entry['name']
        raw_masks[Path(name).stem] = analyze_yamnet(str(Path(FOLDER_MERGED) / name))
        prep_denoise_batch([name], raw_masks)
        raw_masks.clear()  # Only protects pre-denoising; never ducks the separated SFX.
    process_bandit_batch(names)
    for entry in inputs_manifest:
        base = Path(entry['name']).stem
        stem = Path(FOLDER_AI_STEMS) / (base + '_Bandit.wav')
        mix = Path(FOLDER_TC_READY) / (base + '_Mix_Temp.wav')
        finish_v3(str(stem), str(mix))
        info = sf.info(mix)
        if info.frames != entry['frames'] or info.samplerate != 48000 or info.channels != 1:
            raise RuntimeError(f"Durée/canaux modifiés : {entry['name']}")
        pending = Path(FOLDER_TC_READY) / entry['output_name']
        inject_timecode(str(Path(FOLDER_MERGED) / entry['name']), str(mix), str(pending))
        geom = inspect_delivery_wave(pending)
        if any(geom[k] != entry[k] for k in ('frames', 'sample_rate', 'time_reference')):
            raise RuntimeError(f"Géométrie/timecode modifiés : {entry['name']}")
        output = dict(geom, name=entry['output_name'], role=entry['role'],
                      sha256=sha256_file(pending), umid=read_wave_umid(pending))
        dest = run_path / output['name']
        shutil.copy2(pending, dest)
        verify_delivery_file(dest, output)
        manifest['outputs'].append(output)
        print(f"✅ Scène indépendante : {output['name']}")
    # Assert all outputs before publishing the single complete generation pointer.
    if len(manifest['outputs']) != len(inputs_manifest):
        raise RuntimeError('Lot incomplet.')
    verify_unique_wave_umids([run_path / e['name'] for e in manifest['outputs']],
                             forbidden=[e['umid'] for e in inputs_manifest])
    manifest['status'] = 'complete'
    validate_delivery_manifest(manifest)
    write_delivery_manifest(run_path / DELIVERY_MANIFEST, manifest)
    # Do not overwrite another run that started while this one was running.
    current = json.loads(pointer.read_text(encoding='utf-8'))
    if current.get('run_id') != RUN_ID:
        raise RuntimeError('Un autre traitement a remplacé le pointeur; publication refusée.')
    write_delivery_manifest(pointer, manifest)
    WORKFLOW_COMPLETE = True
    print('🎉 Test V3 terminé. Téléchargez avec le bouton ZIP/PTX du frontend MIS À JOUR.')
except Exception:
    manifest['status'] = 'failed'
    current = json.loads(pointer.read_text(encoding='utf-8'))
    if current.get('run_id') == RUN_ID:
        write_delivery_manifest(pointer, manifest)
    raise


In [ ]:
# 8. DÉCONNEXION — uniquement après réussite complète
# Le temps et le tarif Colab doivent être lus dans le compte; aucune estimation fixe.
import time
from google.colab import runtime
if globals().get('WORKFLOW_COMPLETE', False):
    print(f"Lot vérifié sauvegardé dans Drive : {FOLDER_OUT}")
    print("Déconnexion de la VM dans 10 secondes.")
    time.sleep(10)
    runtime.unassign()
else:
    print("Run incomplet : aucune déconnexion automatique. Inspecter l'erreur avant de relancer.")
